# NeuMF — chạy lại số liệu cuối trên Google Colab

Notebook chạy đúng pipeline chính thức của đề tài (PREREG):

| Bước | Script | Ra |
|---|---|---|
| Dữ liệu | `scripts/00_sample_hm.py` | `data/processed/hm/hm500k_transactions.csv` (lấy mẫu tất định theo hash khách hàng) |
| Khám phá và kiểm tra dữ liệu từng bước | các hàm trong `src/data_pipeline/` | thống kê + biểu đồ, không đụng test |
| Đánh giá cuối trên test, 3 seed (chọn số epoch trên val → train lại trên train ∪ val → chấm test) | `scripts/11_final.py` | `outputs/final/seed*/` |
| Kiểm định (Wilcoxon, bootstrap CI, Holm) | `scripts/12_significance.py` | `outputs/final/summary.csv`, `significance.csv` |
| Mở rộng sau khi xem test (PREREG mục 9): late fusion GMF + MLP, BPR-MF + MLP, ItemKNN, UserKNN | `scripts/17_extension.py` | `outputs/final/extension/` |
| Chỉ số @20 từ hạng đã lưu | `scripts/16_extra_k.py` | `outputs/final/extra_k20.csv` |
| Biểu đồ | `scripts/13_plot_final.py` | `outputs/figures/final_*.png` |
| Phân tích phụ: long-tail, cold/warm, beyond-accuracy, Sampled-99, độ trễ | `scripts/14_secondary.py` | `outputs/final/{stratified,beyond_accuracy,sampled99,latency}.csv` |
| Xuất sang báo cáo LaTeX | `scripts/15_export_report.py` | `Report DACNTT/content/tables/*.tex`, `Report DACNTT/media/figures/final/*.png` |
| Đối chiếu câu chữ báo cáo, số chép tay, tái lập so với bản đã commit | `scripts/19_check_report.py` | in ra màn hình |

Cấu hình mô hình lấy từ `audit/best_configs.json` (đã tune chỉ trên val). Notebook **không tune lại**.

**Trước khi chạy:** `Runtime → Change runtime type → GPU (T4)`. Thời gian: khoảng 10–20 phút cho dữ liệu (lần đầu, tải Kaggle), khoảng 1,5–2,5 giờ cho 3 seed (máy gốc RTX 3050 mất khoảng 1,5 giờ
kể cả bước train lại trên train ∪ val; CPU của Colab chậm hơn nên BPR-MF lâu hơn), khoảng 10–15 phút cho phân tích phụ. Giữ tab trình duyệt mở trong lúc chạy để Colab không ngắt phiên.

Thứ tự giống hệt `python run.py final` trên máy local (18 → 11 → 12 → 17 → 16 → 13 → 14 → 15 → 19). Lần chạy gốc dùng RTX 3050: trên T4, Random/MostPopular/ItemKNN/UserKNN phải trùng, BPR-MF gần như trùng, GMF/MLP/NeuMF có thể lệch nhẹ — `19_check_report.py` chỉ ra câu chữ nào cần sửa.

**Khoá test:** `11_final.py` chỉ chạy khi working tree sạch và ghi mỗi lần chấm test vào `audit/test_access_log.csv`.
Chạy xong, làm theo mục 11 để chép kết quả, nhật ký test và bảng/hình báo cáo về repo rồi commit.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi -L || echo "KHÔNG có GPU — vào Runtime > Change runtime type > GPU"

## 2. Cấu hình và mount Google Drive

Drive dùng để: (a) giữ `hm500k_transactions.csv` cho các lần chạy sau, (b) lưu kết quả phòng khi Colab ngắt kết nối.

Repo private: thêm secret `GITHUB_TOKEN` (biểu tượng 🔑 ở thanh bên trái), bật quyền truy cập cho notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/sulinh3625/DU-AN-CNTT.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/neumf_colab"   # nơi lưu dữ liệu + kết quả
REASON = "Chạy lại toàn bộ trên commit cuối (PREREG mục 8, 02/10/2026) — Colab"   # ghi vào test_access_log.csv
SEEDS = "42 2024 2025"
REASON_EXTENSION = REASON + " — mở rộng PREREG mục 9"
REASON_SECONDARY = REASON + " — phân tích phụ"

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
CLONE_URL = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
print("Drive:", DRIVE_DIR, "| token:", "có" if token else "không (repo phải public)")

## 3. Clone repo và cài thư viện

In [ ]:
%cd /content
!rm -rf DU-AN-CNTT
!git clone -q --depth 1 --branch $BRANCH $CLONE_URL DU-AN-CNTT
%cd /content/DU-AN-CNTT/neumf_project
!git log --oneline -1
!pip install -q -r requirements.txt

> Nếu pip báo đã hạ/nâng phiên bản `numpy`/`pandas`: `Runtime → Restart session`, rồi chạy lại ô 2 và dòng `%cd /content/DU-AN-CNTT/neumf_project` (không cần clone lại).

## 4. Dữ liệu hm500k

Ô dưới tự chọn nguồn theo thứ tự:
1. `DRIVE_DIR/hm500k_transactions.csv` nếu đã có (lần chạy trước, hoặc bạn tự upload từ máy).
2. Nếu chưa có: tải `transactions_train.csv` từ Kaggle rồi lấy mẫu bằng `00_sample_hm.py`, sau đó lưu bản hm500k lên Drive.

Để tải từ Kaggle, cần **một trong hai**: secret `KAGGLE_USERNAME` + `KAGGLE_KEY`, hoặc file `DRIVE_DIR/kaggle.json`.
Tài khoản Kaggle phải đã bấm *I Understand and Accept* ở trang cuộc thi
[H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/rules).

In [ ]:
import os, shutil, subprocess
from pathlib import Path

TARGET = Path("data/processed/hm/hm500k_transactions.csv")
CACHED = Path(DRIVE_DIR) / "hm500k_transactions.csv"
TARGET.parent.mkdir(parents=True, exist_ok=True)

if CACHED.exists():
    shutil.copy(CACHED, TARGET)
    print("Dùng bản trên Drive:", CACHED)
else:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        kj = Path(DRIVE_DIR) / "kaggle.json"
        if not kj.exists():
            raise SystemExit("Thiếu thông tin Kaggle: thêm secret KAGGLE_USERNAME/KAGGLE_KEY hoặc đặt kaggle.json vào " + DRIVE_DIR)
        Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
        shutil.copy(kj, Path("~/.kaggle/kaggle.json").expanduser())
        os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
    subprocess.run("pip install -q kaggle", shell=True, check=True)
    raw = Path("data/raw/hm"); raw.mkdir(parents=True, exist_ok=True)
    subprocess.run("kaggle competitions download -c h-and-m-personalized-fashion-recommendations "
                   f"-f transactions_train.csv -p {raw}", shell=True, check=True)
    z = raw / "transactions_train.csv.zip"
    if z.exists():   # kaggle CLI thường trả về .zip
        subprocess.run(f"unzip -o -q {z} -d {raw} && rm {z}", shell=True, check=True)
    subprocess.run("python scripts/00_sample_hm.py", shell=True, check=True)
    shutil.copy(TARGET, CACHED)
    (raw / "transactions_train.csv").unlink()   # 3,5 GB, không cần nữa
    print("Đã lưu hm500k lên Drive:", CACHED)

print(f"{TARGET}: {TARGET.stat().st_size / 1e6:.1f} MB")

## 5. Khám phá và kiểm tra dữ liệu — pipeline từng bước (khoảng 1–2 phút, nên chạy)

Chạy lại đúng các hàm mà `11_final.py` dùng, từng bước, để xem dữ liệu biến đổi ra sao:
load → gộp cặp (user, item) → k-core → re-index → chia theo mốc thời gian chung → negative sampling.
Muốn xem source của hàm nào: `show_source(tên_hàm)`.

Bước 5 so dữ liệu với lần chạy gốc (7.519 user × 10.345 item sau k-core; train 201.801 cặp trên 10.145 item;
train ∪ val 209.212 cặp trên 10.216 item; test 9.229 cặp / 2.996 user). Nếu lệch thì dừng lại, đừng chạy mục 7.

Phần này **không chấm mô hình nào trên test**; chỉ đếm kích thước các tập.

In [ ]:
import sys, inspect
sys.path.insert(0, ".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown

from scripts.common import build_adapter
from src.data_pipeline.preprocessing import aggregate_unique_user_item, reindex_interactions, apply_feedback_weights
from src.data_pipeline.kcore import iterative_k_core
from src.data_pipeline.splitting import global_temporal_split, assert_disjoint_splits, refit_data
from src.data_pipeline.negative_sampling import build_user_positive_sets
from src.data_pipeline.dataset import TrainDataset

def show_source(obj):
    display(Markdown(f"```python\n{inspect.getsource(obj)}\n```"))

cfg, adapter = build_adapter("configs/hm500k_global.yaml")
K, VAL_START, TEST_START = cfg.dataset.k_core, cfg.dataset.val_start, cfg.dataset.test_start
print(f"k-core = {K} | val_start = {VAL_START} | test_start = {TEST_START}")

### Bước 1 — Load và chuẩn hoá cột (`HMAdapter`)

In [ ]:
events = adapter.load_events()
print(f"{len(events):,} giao dịch | {events.user_raw.nunique():,} khách | {events.item_raw.nunique():,} sản phẩm | "
      f"{events.timestamp.min().date()} → {events.timestamp.max().date()}")
fig, ax = plt.subplots(figsize=(11, 3))
events.set_index("timestamp").resample("W").size().plot(ax=ax, color="#2a78d6")
for d, lab in [(VAL_START, "val"), (TEST_START, "test")]:
    ax.axvline(pd.Timestamp(d), color="gray", ls="--"); ax.text(pd.Timestamp(d), ax.get_ylim()[1] * 0.9, f" {lab}")
ax.set_title("Số giao dịch mỗi tuần"); ax.set_xlabel(""); plt.show()
events.head()

### Bước 2 — Gộp giao dịch lặp thành cặp (user, item) duy nhất

In [ ]:
agg = aggregate_unique_user_item(events)
print(f"{len(events):,} giao dịch → {len(agg):,} cặp (giảm {100 * (1 - len(agg) / len(events)):.1f}%)")
print("Số lần mua lặp / cặp:", agg.interaction_count.describe()[["mean", "50%", "max"]].round(2).to_dict())

### Bước 3 — Lọc k-core (độ nhạy theo k)

In [ ]:
rows = []
for k in [1, 3, 5, 10, 15, 20]:
    f = iterative_k_core(agg, k)
    u, i = f.user_raw.nunique(), f.item_raw.nunique()
    rows.append(dict(k=k, users=u, items=i, pairs=len(f), density_pct=100 * len(f) / (u * i) if u and i else 0))
display(pd.DataFrame(rows).style.format({"users": "{:,}", "items": "{:,}", "pairs": "{:,}", "density_pct": "{:.3f}"}))
filtered = iterative_k_core(agg, K)

### Bước 4 — Re-index sang số nguyên 0..N-1

In [ ]:
data = reindex_interactions(filtered)
assert data.df.user.max() == data.n_users - 1 and data.df.item.max() == data.n_items - 1
print(f"n_users = {data.n_users:,} | n_items = {data.n_items:,} | {len(data.df):,} cặp  ✓ index liên tục")

### Bước 5 — Chia theo MỘT mốc thời gian chung

Mỗi cặp xếp theo lần mua đầu: `train < val_start ≤ val < test_start ≤ test`.

- **Val** (chọn cấu hình, số epoch): mô hình học trên train; val chỉ giữ user/item đã có trong train.
- **Test**: trước khi chấm, mô hình được **train lại trên train ∪ val** (mọi cặp trước `test_start`, `refit_data`) đúng số epoch tốt nhất tìm trên val; test chỉ giữ user/item đã có trong train ∪ val.

In [ ]:
tr, va, te = global_temporal_split(data.df, VAL_START, TEST_START)
assert_disjoint_splits(tr, va, te)
trva = refit_data(data.df, TEST_START)   # dữ liệu train lại trước khi chấm test
display(pd.DataFrame([dict(tập=n, cặp=len(x), user=x.user.nunique(), item=x.item.nunique(),
                           từ=x.first_timestamp.min().date(), đến=x.first_timestamp.max().date())
                      for n, x in [("train", tr), ("val", va), ("train ∪ val (train lại)", trva), ("test", te)]]))
print("✓ Không có cặp (user, item) nào trùng giữa các tập")

got = (len(tr), tr.item.nunique(), len(trva), trva.item.nunique(), len(te), te.user.nunique())
want = (201_801, 10_145, 209_212, 10_216, 9_229, 2_996)
if got != want:
    print("⚠ KHÁC lần chạy gốc (train cặp/item, train ∪ val cặp/item, test cặp/user) — kiểm tra file dữ liệu.")
    print("  gốc:", want)
    print("  đây:", got)
else:
    print("✓ Khớp lần chạy gốc: train 201.801 cặp / 10.145 item; train ∪ val 209.212 / 10.216; test 9.229 cặp / 2.996 user")

### Bước 6 — Negative sampling động (`TrainDataset`)

In [ ]:
tr_w, _, _, _ = apply_feedback_weights(tr, va, te, cfg.feedback.mode, cfg.feedback.confidence_alpha)
train_pos = build_user_positive_sets(tr_w, data.n_users)
ds = TrainDataset(tr_w, data.n_items, train_pos, neg_ratio=4, seed=42)
neg = ds.labels == 0
bad = sum(int(i) in train_pos[int(u)] for u, i in zip(ds.users[neg], ds.items[neg]))
print(f"{int((~neg).sum()):,} positive + {int(neg.sum()):,} negative (tỉ lệ 1:4) | negative trùng positive: {bad}",
      "✓" if bad == 0 else "✗")
del events, agg, filtered, data, tr, va, te, trva, tr_w, train_pos, ds

## 6. Kiểm tra trước khi chạy

- Unit test của dự án.
- `scripts/18_preflight.py`: working tree sạch, PREREG đã commit và có mục 9, đủ cấu hình đã tune, dữ liệu khớp lần chạy gốc, tuning mở rộng chạy lại ra đúng cùng số (khoảng 3–5 phút). Báo **CHƯA SẴN SÀNG** thì dừng, đừng chạy mục 7.

In [ ]:
!python -m pytest -q 2>&1 | tail -3

In [ ]:
!python scripts/18_preflight.py

## 7. Đánh giá cuối trên test (3 seed)

Chạy một lần cho cả 3 seed (khoá test yêu cầu tree sạch lúc khởi động; sau seed đầu, `test_access_log.csv` đã bị sửa
nên không chạy tách từng seed được). Log được ghi song song lên Drive.

Nếu Colab ngắt giữa chừng: chạy lại từ ô 2. Mỗi lần chạy lại vẫn được ghi vào nhật ký truy cập test.

In [ ]:
LOG = f"{DRIVE_DIR}/final.log"
!python -u scripts/11_final.py --reason "$REASON" --seeds $SEEDS 2>&1 | tee "$LOG"

## 8. Kiểm định, mở rộng PREREG mục 9, chỉ số @20 và biểu đồ

`17_extension.py` là một lần chấm test (ghi `test_access_log.csv`): nạp GMF, MLP, BPR-MF vừa huấn luyện ở mục 7 để trộn điểm (late fusion), tính ItemKNN/UserKNN trên train ∪ val, kiểm định họ 7 so sánh riêng. Khoảng 5–10 phút.

In [ ]:
import json
done = [int(s) for s in SEEDS.split() if Path(f"outputs/final/seed{s}/results.json").exists()]
missing = sorted(set(map(int, SEEDS.split())) - set(done))
if missing:  # nếu không, 12_significance/ô 10 sẽ đọc summary.csv CŨ đang commit trong repo
    raise SystemExit(f"11_final.py chưa chạy xong seed {missing} — xem log ở mục 7 / {LOG}")
for s in done:
    r = json.loads(Path(f"outputs/final/seed{s}/results.json").read_text(encoding="utf-8"))
    print(f"seed {s}: commit {r['provenance']['git_commit'][:7]} | {r['n_test_users']:,} user test")
!python scripts/12_significance.py
!python -u scripts/17_extension.py --reason "$REASON_EXTENSION" 2>&1 | tee "$DRIVE_DIR/extension.log"
if not Path("outputs/final/extension/significance.csv").exists():  # tee nuốt mã lỗi -> tự kiểm tra đầu ra
    raise SystemExit(f"17_extension.py lỗi — xem {DRIVE_DIR}/extension.log")
!python scripts/16_extra_k.py
!python scripts/13_plot_final.py

## 9. Phân tích phụ (khoảng 10–15 phút)

Nạp lại checkpoint của mục 7 (**không train lại**) và tính: NDCG@10 theo nhóm sản phẩm head/tail và nhóm user cold/warm, coverage/novelty/ARP/HRR của top-10, Sampled-99 (đối chiếu protocol NCF gốc) và độ trễ top-10 trên CPU.

Đây cũng là một lần chấm test nên được ghi 1 dòng vào `audit/test_access_log.csv` với lý do `REASON_SECONDARY`. Script chỉ đòi code/cấu hình không bị sửa (cho phép `outputs/` và nhật ký test do mục 7–8 vừa ghi).

> Độ trễ đo trên CPU của máy Colab (thường 2 vCPU) — ghi rõ cấu hình này khi trích trong báo cáo.

In [ ]:
!python -u scripts/14_secondary.py --reason "$REASON_SECONDARY" 2>&1 | tee "$DRIVE_DIR/secondary.log"
need = ["stratified.csv", "beyond_accuracy.csv", "sampled99.csv", "latency.csv"]
lack = [n for n in need if not Path(f"outputs/final/{n}").exists()]
if lack:  # tee nuốt mã lỗi của python -> tự kiểm tra đầu ra
    raise SystemExit(f"14_secondary.py lỗi, thiếu {lack} — xem {DRIVE_DIR}/secondary.log")

## 10. Xuất số liệu sang báo cáo LaTeX và đối chiếu câu chữ

Sinh bảng `.tex`, macro số liệu (`results_macros.tex`, gồm cả câu kết luận tự sinh của họ so sánh mở rộng) và chép hình vào `Report DACNTT/` của bản clone. `19_check_report.py` so số mới với bản đã commit, kiểm tra các khẳng định bằng chữ ở Chương 1, 4, 5 và phần tóm tắt, và liệt kê dòng nào trong `van_dap.md`/tài liệu còn ghi số cũ — lưu log lên Drive để sửa trên máy.

In [ ]:
!python scripts/15_export_report.py
!python scripts/19_check_report.py 2>&1 | tee "$DRIVE_DIR/check_report.log"

## 11. Lưu kết quả lên Drive và tải về máy

In [ ]:
import shutil, time
from pathlib import Path

stamp = time.strftime("%Y%m%d-%H%M")
dest = Path(DRIVE_DIR) / f"results_{stamp}"
shutil.copytree("outputs/final", dest / "final", dirs_exist_ok=True)  # gồm cả checkpoint (*.pt, bpr.npz; ~60 MB) — 14_secondary và repo cần
shutil.copytree("outputs/figures", dest / "figures", dirs_exist_ok=True)
shutil.copy("audit/test_access_log.csv", dest / "test_access_log.csv")
if Path(f"{DRIVE_DIR}/check_report.log").exists():
    shutil.copy(f"{DRIVE_DIR}/check_report.log", dest / "check_report.log")
report = Path("../Report DACNTT")
shutil.copytree(report / "content" / "tables", dest / "report" / "content" / "tables", dirs_exist_ok=True)
if (report / "media" / "figures" / "final").exists():
    shutil.copytree(report / "media" / "figures" / "final", dest / "report" / "media" / "figures" / "final",
                    dirs_exist_ok=True)
archive = shutil.make_archive(f"/content/neumf_results_{stamp}", "zip", dest)
print("Drive:", dest)

from google.colab import files
files.download(archive)

Bản lưu đã gồm checkpoint (`*.pt`, `bpr.npz`) của `outputs/final/seed*/`; repo cũng theo dõi các file này trong git
(xem `.gitignore`), nên chép nguyên thư mục `final/` về là đủ.

### Chép về repo trên máy (sau khi giải nén file zip)

| Trong zip | Chép vào repo |
|---|---|
| `final/` (gồm `seed*/` kèm checkpoint, `extension/`, `summary.csv`, `significance.csv`, `extra_k20.csv`, `stratified.csv`, `beyond_accuracy.csv`, `sampled99.csv`, `latency.csv`) | `neumf_project/outputs/final/` |
| `figures/final_*.png` | `neumf_project/outputs/figures/` |
| `test_access_log.csv` | `neumf_project/audit/test_access_log.csv` (ghi đè — file đã gồm dòng cũ + dòng mới) |
| `report/` | `Report DACNTT/` (ghi đè `content/tables/` và `media/figures/final/`) |

Rồi chạy `python run.py check-report` trên máy (giống `check_report.log`): sửa các câu báo `[SAI]` và các dòng tài liệu còn ghi số cũ, biên dịch lại báo cáo (`Report DACNTT/compile.bat`) và commit.

## 12. Xem nhanh kết quả

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary = pd.read_csv("outputs/final/summary.csv", index_col=0)
cols = [c for c in summary.columns if c.endswith("_mean")]
display(summary[cols].round(5))

sig = pd.read_csv("outputs/final/significance.csv")
display(sig[["A", "B", "mean_A", "mean_B", "rel_diff", "ci_low", "ci_high", "p_wilcoxon", "p_holm", "verdict"]].round(5))

for name in ("stratified", "beyond_accuracy", "sampled99", "latency"):
    f = Path(f"outputs/final/{name}.csv")
    if f.exists():
        df = pd.read_csv(f)
        if "seed" in df.columns:
            keys = [c for c in ("model", "subset") if c in df.columns]
            df = df.groupby(keys).mean(numeric_only=True).drop(columns="seed")
        print(name); display(df.round(5))

for f in sorted(Path("outputs/figures").glob("final_*.png")):
    print(f.name); display(Image(str(f), width=900))

### Đối chiếu với lần chạy gốc (RTX 3050, commit `c559c86`, có bước train lại trên train ∪ val)

NDCG@10 trung bình 3 seed (2.996 user test): BPR-MF 0,01007 · NeuMF-Pretrained 0,00957 · MLP 0,00931 · MostPopular 0,00917 ·
NeuMF-Scratch 0,00910 · GMF 0,00907 · Random 0,00065. Không so sánh nào có ý nghĩa sau hiệu chỉnh Holm.

Random và MostPopular phải ra **giống hệt**. BPR-MF (numpy) gần như giống hệt (thư viện BLAS khác máy có thể lệch cực nhỏ).
GMF/MLP/NeuMF trên GPU khác (T4 so với RTX 3050) có thể lệch nhẹ do phép tính cuDNN không tất định. Kết luận thống kê mới là thứ cần giữ.

Phần mở rộng (late fusion, ItemKNN, UserKNN) chưa có số test trước lần chạy này; tham số chọn trên val: UserKNN k = 200 (val NDCG@10 0,01340), ItemKNN k = 100, shrink = 50 (0,01217), late fusion GMF + MLP w = 0,1 (0,01005), BPR-MF + MLP w = 0,3 (0,00972) — `audit/PREREG.md` mục 9.